In [4]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

data = load_breast_cancer()
X = data.data
y = data.target


scaler = StandardScaler()
X = scaler.fit_transform(X)


X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


n_nests = 10
pa = 0.25
iterations = 30
alpha = 0.01
beta = 0.5
n_features = X.shape[1]


def binary_solution(solution):
    return (1 / (1 + np.exp(-solution)) > 0.5).astype(int)


def fitness(solution):
    selected = binary_solution(solution)

    if np.sum(selected) == 0:
        return 1.0

    model = KNeighborsClassifier(n_neighbors=5)
    model.fit(X_train[:, selected == 1], y_train)

    pred = model.predict(X_test[:, selected == 1])
    error = 1 - accuracy_score(y_test, pred)

    feature_ratio = np.sum(selected) / n_features

    return beta * error + (1 - beta) * feature_ratio


def levy_flight():
    u = np.random.normal(0, 1)
    v = np.random.normal(0, 1)
    return u / (abs(v) ** (1 / 1.5))


nests = np.random.uniform(-1, 1, (n_nests, n_features))

scores = np.array([fitness(n) for n in nests])


for t in range(iterations):

    for i in range(n_nests):

        
        new_nest = nests[i] + alpha * levy_flight() * np.random.randn(n_features)

        new_nest = np.clip(new_nest, -5, 5)
        new_score = fitness(new_nest)

        
        j = np.random.randint(n_nests)

        if new_score < scores[j]:
            nests[j] = new_nest
            scores[j] = new_score

    
    n_abandon = max(1, int(pa * n_nests))
    worst = np.argsort(scores)[-n_abandon:]

    nests[worst] = np.random.uniform(
        -1, 1, (n_abandon, n_features)
    )

    scores[worst] = np.array(
        [fitness(n) for n in nests[worst]]
    )

    print("Iteration:", t + 1,
          "Best Fitness:", round(np.min(scores), 4))


best_index = np.argmin(scores)
best_features = binary_solution(nests[best_index])

selected_features = np.where(best_features == 1)[0]


model = KNeighborsClassifier(n_neighbors=5)
model.fit(X_train[:, selected_features], y_train)

predictions = model.predict(X_test[:, selected_features])
accuracy = accuracy_score(y_test, predictions)

print("\n----- FINAL RESULTS -----")
print("Total Features:", n_features)
print("Selected Features:", len(selected_features))
print("Selected Feature Indices:", selected_features)
print("Classification Accuracy:", round(accuracy * 100, 2), "%")
print("Best Fitness:", round(scores[best_index], 4))


Iteration: 1 Best Fitness: 0.2351
Iteration: 2 Best Fitness: 0.2351
Iteration: 3 Best Fitness: 0.2351
Iteration: 4 Best Fitness: 0.2351
Iteration: 5 Best Fitness: 0.2298
Iteration: 6 Best Fitness: 0.2298
Iteration: 7 Best Fitness: 0.2298
Iteration: 8 Best Fitness: 0.2298
Iteration: 9 Best Fitness: 0.2298
Iteration: 10 Best Fitness: 0.2132
Iteration: 11 Best Fitness: 0.2053
Iteration: 12 Best Fitness: 0.1965
Iteration: 13 Best Fitness: 0.1965
Iteration: 14 Best Fitness: 0.1842
Iteration: 15 Best Fitness: 0.1842
Iteration: 16 Best Fitness: 0.1842
Iteration: 17 Best Fitness: 0.1842
Iteration: 18 Best Fitness: 0.1842
Iteration: 19 Best Fitness: 0.1632
Iteration: 20 Best Fitness: 0.1632
Iteration: 21 Best Fitness: 0.1632
Iteration: 22 Best Fitness: 0.1632
Iteration: 23 Best Fitness: 0.1632
Iteration: 24 Best Fitness: 0.1632
Iteration: 25 Best Fitness: 0.1632
Iteration: 26 Best Fitness: 0.1632
Iteration: 27 Best Fitness: 0.1632
Iteration: 28 Best Fitness: 0.1632
Iteration: 29 Best Fitness: 0